In [1]:
using CSV, DataFrames, Statistics

# === INPUT DATA ===
# Replace with your HPLC replicate data
# Each row corresponds to triplicate measurements for one sample
data = [
    [3939.237, 3710.146, 3961.150],
    [20020.892, 19430.791, 32769.866],
    [4117.070, 4128.740, 4274.476],
    [21173.682, 21301.682, 22729.838],
    [4873.346, 4918.758, 4995.244],
    [27273.054, 27575.705, 27910.514],
    [4918.758, 4941.586, 4964.414],
    [31077.958, 31970.035, 31191.336],
    [4738.029, 4747.114, 4756.200],
    [34690.074, 35191.039, 34912.278],
    [4557.382, 4584.938, 4612.495],
    [37773.254, 38340.109, 37990.559]
]

# === CALCULATIONS ===
means = [mean(row) for row in data]
stds = [std(row) for row in data]
rsd = [100 * std(row) / mean(row) for row in data]

df = DataFrame(
    Sample = 1:length(data),
    Mean = means,
    StdDev = stds,
    RSD_percent = rsd
)

# === SUMMARY ===
overall_std = mean(stds)
overall_rsd = mean(rsd)

println("Per-sample summary:")
println(df)
println("\nAverage absolute uncertainty (σ̄): ", round(overall_std, digits=3))
println("Average relative uncertainty (RSD̄): ", round(overall_rsd, digits=2), " %")



Per-sample summary:
12×4 DataFrame
 Row │ Sample  Mean      StdDev     RSD_percent 
     │ Int64   Float64   Float64    Float64     
─────┼──────────────────────────────────────────
   1 │      1   3870.18   139.024      3.59218
   2 │      2  24073.8   7536.75      31.3068
   3 │      3   4173.43    87.7039     2.10148
   4 │      4  21735.1    863.871      3.97455
   5 │      5   4929.12    61.6056     1.24983
   6 │      6  27586.4    318.865      1.15588
   7 │      7   4941.59    22.828      0.461957
   8 │      8  31413.1    485.632      1.54595
   9 │      9   4747.11     9.0855     0.19139
  10 │     10  34931.1    251.014      0.718597
  11 │     11   4584.94    27.5565     0.601022
  12 │     12  38034.6    285.987      0.751912

Average absolute uncertainty (σ̄): 840.827
Average relative uncertainty (RSD̄): 3.97 %


In [4]:
using Statistics, DataFrames, Distributions

# === INPUT DATA ===
data = [
    [3939.237, 3710.146, 3961.150],
    [20020.892, 19430.791, 32769.866],
    [4117.070, 4128.740, 4274.476],
    [21173.682, 21301.682, 22729.838],
    [4873.346, 4918.758, 4995.244],
    [27273.054, 27575.705, 27910.514],
    [4918.758, 4941.586, 4964.414],
    [31077.958, 31970.035, 31191.336],
    [4738.029, 4747.114, 4756.200],
    [34690.074, 35191.039, 34912.278],
    [4557.382, 4584.938, 4612.495],
    [37773.254, 38340.109, 37990.559]
]

# === GRUBBS TEST ===
function grubbs_test(x::Vector{Float64}; α=0.05)
    n = length(x)
    x̄ = mean(x)
    s = std(x)
    G = maximum(abs.(x .- x̄)) / s
    tcrit = quantile(TDist(n - 2), 1 - α / (2n))
    Gcrit = ((n - 1) / sqrt(n)) * sqrt(tcrit^2 / (n - 2 + tcrit^2))
    i_outlier = argmax(abs.(x .- x̄))
    return (i_outlier, G > Gcrit, G, Gcrit)
end

# === DIXON Q TEST ===
function dixons_q_test(x::Vector{Float64}; α=0.05)
    n = length(x)
    sorted = sort(x)
    # Critical Q values for small n
    Qcrit = Dict(
        3 => α == 0.05 ? 0.970 : 0.941,
        4 => α == 0.05 ? 0.829 : 0.765,
        5 => α == 0.05 ? 0.710 : 0.642
    )[n]
    Q_high = (sorted[end] - sorted[end-1]) / (sorted[end] - sorted[1])
    Q_low  = (sorted[2] - sorted[1]) / (sorted[end] - sorted[1])
    if Q_high > Q_low
        return (findfirst(==(sorted[end]), x), Q_high > Qcrit, Q_high, Qcrit)
    else
        return (findfirst(==(sorted[1]), x), Q_low > Qcrit, Q_low, Qcrit)
    end
end

# === APPLY BOTH TESTS ===
results = DataFrame(Sample=Int[], Mean=Float64[],
    GrubbsIndex=Int[], GrubbsFlag=Bool[], G=Float64[], Gcrit=Float64[],
    DixonIndex=Int[], DixonFlag=Bool[], Q=Float64[], Qcrit=Float64[],
    CombinedFlag=Bool[])

for (i, row) in enumerate(data)
    μ = mean(row)

    # Grubbs
    gi, gflag, G, Gcrit = grubbs_test(row; α=0.05)

    # Dixon
    di, dflag, Q, Qcrit = dixons_q_test(row; α=0.05)

    # Combined flag if either test detects an outlier
    combined = gflag || dflag

    push!(results, (i, μ, gi, gflag, G, Gcrit, di, dflag, Q, Qcrit, combined))
end

println("=== Outlier Detection Results ===")
println(results)

println("\nFlagged samples:")
for i in 1:size(results, 1)
    if results.CombinedFlag[i]
        val_g = data[i][results.GrubbsIndex[i]]
        val_d = data[i][results.DixonIndex[i]]
        println("Sample $(results.Sample[i]): outlier candidates → Grubbs = $(round(val_g,digits=3)), Dixon = $(round(val_d,digits=3))")
    end
end




=== Outlier Detection Results ===
12×11 DataFrame
 Row │ Sample  Mean      GrubbsIndex  GrubbsFlag  G        Gcrit    DixonIndex  DixonFlag  Q         Qcrit    CombinedFlag 
     │ Int64   Float64   Int64        Bool        Float64  Float64  Int64       Bool       Float64   Float64  Bool         
─────┼─────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │      1   3870.18            2       false  1.15111   1.1543           2      false  0.912699     0.97         false
   2 │      2  24073.8             3       false  1.15382   1.1543           3      false  0.955761     0.97         false
   3 │      3   4173.43            3       false  1.15214   1.1543           3      false  0.925861     0.97         false
   4 │      4  21735.1             3       false  1.15153   1.1543           3      false  0.917746     0.97         false
   5 │      5   4929.12            3       false  1.07341   1.1543           3      fa